In [1]:
# CELL 1 — Install all required libraries
!pip install langchain langchain-community langchain-google-genai chromadb pypdf groq langgraph ragas datasets wikipedia-api tenacity -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0

In [2]:
# CELL 2 — Imports and API key setup
try:
    import os
    import time
    import warnings
    warnings.filterwarnings("ignore")

    from google.colab import userdata

    # Load API keys from Colab Secrets
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")
    os.environ["GROQ_API_KEY"]   = userdata.get("GROQ_KEY")

    print("✅ API keys loaded successfully")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except KeyError as e:
    print(f"❌ Missing Key Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ API keys loaded successfully


In [3]:
# CELL 3 — Fetch Wikipedia articles as document corpus (10-20 documents)

try:
    import wikipediaapi
    import os

    wiki = wikipediaapi.Wikipedia(
        language="en",
        user_agent="RAGInternshipBot/1.0 (internship project)"
    )

    # 12 AI/ML Wikipedia articles — your document set
    TOPICS = [
        "Retrieval-augmented generation",
        "Large language model",
        "Transformer (deep learning architecture)",
        "Word embedding",
        "Convolutional neural network",
        "Recurrent neural network",
        "Attention (machine learning)",
        "BERT (language model)",
        "GPT-4",
        "Knowledge graph",
        "Vector database",
        "Reinforcement learning from human feedback",
    ]

    os.makedirs("/content/docs", exist_ok=True)

    fetched = []
    for topic in TOPICS:
        try:
            page = wiki.page(topic)

            if page.exists():
                safe_name = topic.replace("/", "_").replace(" ", "_")
                filepath = f"/content/docs/{safe_name}.txt"

                with open(filepath, "w", encoding="utf-8") as f:
                    # Write first 8000 chars to keep chunks manageable
                    f.write(page.text[:8000])

                fetched.append(filepath)
                print(f"✅ Saved: {topic}")

            else:
                print(f"⚠️ Not found: {topic}")

        except Exception as e:
            print(f"❌ Error processing '{topic}': {e}")

    print(f"\n📚 Total documents fetched: {len(fetched)}")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except PermissionError as e:
    print(f"❌ Permission Error: {e}")

except OSError as e:
    print(f"❌ File System Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Saved: Retrieval-augmented generation
✅ Saved: Large language model
✅ Saved: Transformer (deep learning architecture)
✅ Saved: Word embedding
✅ Saved: Convolutional neural network
✅ Saved: Recurrent neural network
✅ Saved: Attention (machine learning)
✅ Saved: BERT (language model)
✅ Saved: GPT-4
✅ Saved: Knowledge graph
✅ Saved: Vector database
✅ Saved: Reinforcement learning from human feedback

📚 Total documents fetched: 12


In [4]:
# CELL 4 — Ingestion: Load → Chunk → Embed (Gemini) → Store (ChromaDB)

try:
    import os
    import time
    from langchain_community.document_loaders import TextLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from langchain_community.vectorstores import Chroma
    from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
    import google.api_core.exceptions

    # ── Step 1: Load all documents ──────────────────────────────────────────────
    all_docs = []
    doc_dir = "/content/docs"

    for filename in os.listdir(doc_dir):
        if filename.endswith(".txt"):
            try:
                filepath = os.path.join(doc_dir, filename)
                loader = TextLoader(filepath, encoding="utf-8")
                docs = loader.load()

                # Attach source metadata
                for doc in docs:
                    doc.metadata["source"] = filename.replace(".txt", "").replace("_", " ")

                all_docs.extend(docs)
                print(f"📄 Loaded: {filename}  ({len(docs[0].page_content)} chars)")

            except Exception as e:
                print(f"❌ Error loading '{filename}': {e}")

    print(f"\n✅ Total documents loaded: {len(all_docs)}")

    # ── Step 2: Split into ~500-word chunks ────────────────────────────────────
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    chunks = splitter.split_documents(all_docs)
    print(f"✅ Total chunks created: {len(chunks)}")

    # Preview first chunk
    try:
        print(f"\n🔍 Sample chunk:\n{chunks[0].page_content[:300]}...")
        print(f"   Metadata: {chunks[0].metadata}")
    except IndexError:
        print("⚠️ No chunks available to preview.")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except FileNotFoundError as e:
    print(f"❌ File Not Found Error: {e}")

except PermissionError as e:
    print(f"❌ Permission Error: {e}")

except OSError as e:
    print(f"❌ Operating System Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

📄 Loaded: Large_language_model.txt  (8000 chars)
📄 Loaded: BERT_(language_model).txt  (8000 chars)
📄 Loaded: Retrieval-augmented_generation.txt  (8000 chars)
📄 Loaded: Vector_database.txt  (3650 chars)
📄 Loaded: Recurrent_neural_network.txt  (8000 chars)
📄 Loaded: Word_embedding.txt  (8000 chars)
📄 Loaded: Convolutional_neural_network.txt  (8000 chars)
📄 Loaded: Attention_(machine_learning).txt  (8000 chars)
📄 Loaded: Knowledge_graph.txt  (8000 chars)
📄 Loaded: Transformer_(deep_learning_architecture).txt  (8000 chars)
📄 Loaded: Reinforcement_learning_from_human_feedback.txt  (8000 chars)
📄 Loaded: GPT-4.txt  (8000 chars)

✅ Total documents loaded: 12
✅ Total chunks created: 304

🔍 Sample chunk:
A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contexts, and are a foundational technology behind modern chatbots

In [5]:
# CELL 5 — Embed with google-generativeai SDK directly (using models/gemini-embedding-001)

try:
    import os
    import time
    import google.generativeai as genai
    import chromadb
    import builtins

    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

    CHROMA_DIR = "/content/chroma_db"
    EMBEDDING_MODEL = "models/gemini-embedding-001"   # ← confirmed from diagnostic

    # ── Custom embedding class using SDK directly (bypasses langchain v1beta bug) ─
    class GeminiEmbeddings:
        def __init__(self, model_name: str):
            self.model_name = model_name

        def embed_documents(self, texts: list) -> list:
            vectors = []
            for text in texts:
                try:
                    result = genai.embed_content(
                        model=self.model_name,
                        content=text,
                        task_type="retrieval_document"
                    )
                    vectors.append(result["embedding"])

                except Exception as e:
                    print(f"❌ Error embedding document: {e}")
                    raise

            return vectors

        def embed_query(self, text: str) -> list:
            try:
                result = genai.embed_content(
                    model=self.model_name,
                    content=text,
                    task_type="retrieval_query"
                )
                return result["embedding"]

            except Exception as e:
                print(f"❌ Error embedding query: {e}")
                raise

    embeddings = GeminiEmbeddings(EMBEDDING_MODEL)

    # ── Quick sanity test ─────────────────────────────────────────────────────────
    try:
        print("🔍 Testing embedding model …")
        test_vec = embeddings.embed_query("hello world")
        print(f"✅ Embedding works! Vector dimension: {len(test_vec)}")

    except Exception as e:
        print(f"❌ Embedding model test failed: {e}")
        raise

    # ── Batch embed + store in ChromaDB ──────────────────────────────────────────
    BATCH_SIZE = 20

    def embed_in_batches(chunks, embeddings_obj, persist_dir, batch_size=20):
        try:
            client = chromadb.PersistentClient(path=persist_dir)

            # Clear existing collection if re-running cell
            try:
                client.delete_collection("rag_documents")
                print("  🗑️  Cleared existing collection")
            except Exception:
                pass

            collection = client.create_collection(
                name="rag_documents",
                metadata={"hnsw:space": "cosine"}
            )

            total = len(chunks)
            doc_id = 0

            for i in range(0, total, batch_size):
                batch = chunks[i: i + batch_size]

                print(
                    f"  Batch {i//batch_size + 1}/{(total + batch_size - 1)//batch_size} "
                    f"(chunks {i+1}–{min(i+batch_size, total)} of {total})",
                    end=" ... "
                )

                for attempt in range(5):
                    try:
                        texts = [c.page_content for c in batch]
                        metadatas = [c.metadata for c in batch]
                        ids = [f"doc_{doc_id + j}" for j in range(len(batch))]
                        vectors = embeddings_obj.embed_documents(texts)

                        collection.add(
                            documents=texts,
                            embeddings=vectors,
                            metadatas=metadatas,
                            ids=ids
                        )

                        doc_id += len(batch)
                        print("✅")
                        break

                    except Exception as e:
                        err_str = str(e)

                        if (
                            "429" in err_str
                            or "RESOURCE_EXHAUSTED" in err_str
                            or "quota" in err_str.lower()
                        ):
                            wait = 60 * (attempt + 1)
                            print(
                                f"\n  ⚠️  Quota hit — waiting {wait}s (retry {attempt+1}/5)",
                                end=" "
                            )
                            time.sleep(wait)

                        else:
                            print(f"\n  ❌ Unexpected error: {e}")
                            raise

                time.sleep(1)   # polite pause between batches

            return collection, client

        except Exception as e:
            print(f"❌ Error during batch embedding/storage: {e}")
            raise

    print(f"\n🚀 Starting ingestion — {len(chunks)} chunks into ChromaDB …\n")

    collection, chroma_client = embed_in_batches(
        chunks,
        embeddings,
        CHROMA_DIR,
        BATCH_SIZE
    )

    print(f"\n✅ ChromaDB built at     : {CHROMA_DIR}")
    print(f"✅ Total chunks stored   : {collection.count()}")

    # ── Save to builtins so later cells can reuse without re-embedding ────────────
    try:
        builtins.gemini_embeddings = embeddings
        builtins.chroma_collection = collection
        builtins.chroma_client_obj = chroma_client
        builtins.embedding_model_name = EMBEDDING_MODEL

        print(f"\n✅ Saved to builtins — ready for Cell 6, 7, 8")

    except Exception as e:
        print(f"❌ Error saving objects to builtins: {e}")
        raise

except ImportError as e:
    print(f"❌ Import Error: {e}")

except KeyError as e:
    print(f"❌ Missing Environment Variable: {e}")

except FileNotFoundError as e:
    print(f"❌ File Not Found Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

🔍 Testing embedding model …
✅ Embedding works! Vector dimension: 3072

🚀 Starting ingestion — 304 chunks into ChromaDB …

  Batch 1/16 (chunks 1–20 of 304) ... 

ERROR:tornado.access:503 POST /v1beta/models/gemini-embedding-001:embedContent?%24alt=json%3Benum-encoding%3Dint (::1) 379.90ms


✅
  Batch 2/16 (chunks 21–40 of 304) ... ✅
  Batch 3/16 (chunks 41–60 of 304) ... ✅
  Batch 4/16 (chunks 61–80 of 304) ... ✅
  Batch 5/16 (chunks 81–100 of 304) ... ✅
  Batch 6/16 (chunks 101–120 of 304) ... ✅
  Batch 7/16 (chunks 121–140 of 304) ... ✅
  Batch 8/16 (chunks 141–160 of 304) ... ✅
  Batch 9/16 (chunks 161–180 of 304) ... ✅
  Batch 10/16 (chunks 181–200 of 304) ... ✅
  Batch 11/16 (chunks 201–220 of 304) ... ✅
  Batch 12/16 (chunks 221–240 of 304) ... ✅
  Batch 13/16 (chunks 241–260 of 304) ... ✅
  Batch 14/16 (chunks 261–280 of 304) ... ✅
  Batch 15/16 (chunks 281–300 of 304) ... ✅
  Batch 16/16 (chunks 301–304 of 304) ... ✅

✅ ChromaDB built at     : /content/chroma_db
✅ Total chunks stored   : 304

✅ Saved to builtins — ready for Cell 6, 7, 8


In [6]:
# CELL 6 — Query: retrieve top-5 chunks + generate answer with Groq

try:
    import chromadb
    from groq import Groq
    import os
    import builtins

    CHROMA_DIR = "/content/chroma_db"

    # ── Load collection + embeddings ──────────────────────────────────────────────
    chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
    collection = chroma_client.get_collection("rag_documents")
    embeddings = builtins.gemini_embeddings

    groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def retrieve_chunks(question: str, k: int = 5) -> dict:
        """Embed the question and retrieve top-k chunks from ChromaDB."""
        try:
            query_vec = embeddings.embed_query(question)

            results = collection.query(
                query_embeddings=[query_vec],
                n_results=k,
                include=["documents", "metadatas", "distances"]
            )

            return results

        except Exception as e:
            print(f"❌ Error retrieving chunks: {e}")
            raise

    def generate_answer(question: str, documents: list, metadatas: list) -> str:
        """Build prompt from retrieved chunks and call Groq."""
        try:
            context = "\n\n---\n\n".join(
                [
                    f"[Source: {m.get('source', 'Unknown')}]\n{doc}"
                    for doc, m in zip(documents, metadatas)
                ]
            )

            prompt = f"""You are a helpful AI assistant. Answer the question using ONLY the context provided below.
Cite the source for every fact you state. If the context does not contain enough information, say so clearly.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=512
            )

            return response.choices[0].message.content

        except Exception as e:
            print(f"❌ Error generating answer: {e}")
            raise

    def ask_question(question: str, k: int = 5) -> dict:
        """Full RAG pipeline: embed → retrieve → generate."""
        try:
            results = retrieve_chunks(question, k=k)

            documents = results["documents"][0]
            metadatas = results["metadatas"][0]

            answer = generate_answer(question, documents, metadatas)

            sources = [m.get("source", "Unknown") for m in metadatas]

            return {
                "question": question,
                "answer": answer,
                "chunks": documents,
                "sources": sources
            }

        except Exception as e:
            print(f"❌ Error in RAG pipeline: {e}")
            raise

    # ── Test with a sample question ───────────────────────────────────────────────
    try:
        print("🚀 Testing RAG query pipeline …\n")

        result = ask_question("What is retrieval-augmented generation?")

        print(f"❓ Question : {result['question']}")
        print(f"\n💬 Answer  :\n{result['answer']}")
        print(f"\n📎 Sources  : {list(set(result['sources']))}")
        print(f"\n📄 Chunks   : {len(result['chunks'])} retrieved")

    except Exception as e:
        print(f"❌ Test query failed: {e}")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except KeyError as e:
    print(f"❌ Missing Environment Variable: {e}")

except AttributeError as e:
    print(f"❌ Builtins Object Error: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

🚀 Testing RAG query pipeline …

❓ Question : What is retrieval-augmented generation?

💬 Answer  :
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources [Source: Retrieval-augmented generation]. It combines a parametric language model with a non-parametric external memory accessed through retrieval at inference time [Source: Retrieval-augmented generation]. RAG enhances large language models by incorporating an information-retrieval mechanism that allows models to access and utilize additional data beyond their original training set [Source: Retrieval-augmented generation].

📎 Sources  : ['Vector database', 'Retrieval-augmented generation']

📄 Chunks   : 5 retrieved


In [7]:
# CELL 7 — LangGraph pipeline (retrieve → generate → error_handler)

try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict, List, Optional
    from groq import Groq
    import chromadb
    import os
    import builtins

    # ── State schema ──────────────────────────────────────────────────────────────
    class RAGState(TypedDict):
        question: str
        chunks: List[str]
        sources: List[str]
        answer: str
        error: Optional[str]
        vectorstore: Optional[object]

    # ── Shared resources ──────────────────────────────────────────────────────────
    _embeddings = builtins.gemini_embeddings
    _chroma_client = chromadb.PersistentClient(path="/content/chroma_db")
    _collection = _chroma_client.get_collection("rag_documents")
    _groq = Groq(api_key=os.environ["GROQ_API_KEY"])

    # ── Node 1: Retrieve ──────────────────────────────────────────────────────────
    def retrieve_node(state: RAGState) -> RAGState:
        """Retrieve top-5 relevant chunks from ChromaDB."""
        try:
            query_vec = _embeddings.embed_query(state["question"])

            results = _collection.query(
                query_embeddings=[query_vec],
                n_results=5,
                include=["documents", "metadatas"]
            )

            docs = results["documents"][0]
            metas = results["metadatas"][0]

            return {
                **state,
                "chunks": docs,
                "sources": [m.get("source", "Unknown") for m in metas],
                "error": None
            }

        except Exception as e:
            return {
                **state,
                "chunks": [],
                "sources": [],
                "error": f"Retrieval error: {e}"
            }

    # ── Node 2: Generate ──────────────────────────────────────────────────────────
    def generate_node(state: RAGState) -> RAGState:
        """Generate answer using Groq based on retrieved chunks."""

        if state.get("error"):
            return state

        if not state["chunks"]:
            return {
                **state,
                "answer": "No relevant chunks found.",
                "error": "No chunks retrieved."
            }

        context = "\n\n---\n\n".join(
            [
                f"[Source: {src}]\n{chunk}"
                for src, chunk in zip(state["sources"], state["chunks"])
            ]
        )

        prompt = f"""You are a helpful AI assistant. Answer the question using ONLY the context provided below.
Cite the source for every fact you state. If the context does not contain enough information, say so clearly.

CONTEXT:
{context}

QUESTION: {state['question']}

ANSWER:"""

        try:
            response = _groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=512
            )

            return {
                **state,
                "answer": response.choices[0].message.content,
                "error": None
            }

        except Exception as e:
            return {
                **state,
                "answer": "",
                "error": f"Generation error: {e}"
            }

    # ── Node 3: Error handler ─────────────────────────────────────────────────────
    def error_handler_node(state: RAGState) -> RAGState:
        """Handle errors gracefully and return a user-friendly message."""
        try:
            return {
                **state,
                "answer": f"⚠️ Pipeline error: {state.get('error', 'Unknown error')}. "
                          "Please check your API keys and document store."
            }

        except Exception as e:
            return {
                **state,
                "answer": f"⚠️ Error handler failed: {e}"
            }

    # ── Router: decide whether to generate or handle error ───────────────────────
    def route_after_retrieval(state: RAGState) -> str:
        try:
            return "error_handler" if state.get("error") else "generate"

        except Exception:
            return "error_handler"

    # ── Build the LangGraph ───────────────────────────────────────────────────────
    try:
        builder = StateGraph(RAGState)

        builder.add_node("retrieve", retrieve_node)
        builder.add_node("generate", generate_node)
        builder.add_node("error_handler", error_handler_node)

        builder.set_entry_point("retrieve")

        builder.add_conditional_edges(
            "retrieve",
            route_after_retrieval,
            {
                "generate": "generate",
                "error_handler": "error_handler"
            }
        )

        builder.add_edge("generate", END)
        builder.add_edge("error_handler", END)

        rag_pipeline = builder.compile()

        print("✅ LangGraph RAG pipeline compiled successfully")
        print("   Nodes : retrieve → (generate | error_handler) → END\n")

    except Exception as e:
        print(f"❌ Pipeline compilation failed: {e}")
        raise

    # ── Run a test question through the pipeline ──────────────────────────────────
    try:
        print("=" * 60)
        print("🚀 Running LangGraph pipeline …")
        print("=" * 60 + "\n")

        initial_state: RAGState = {
            "question": "What is a large language model?",
            "chunks": [],
            "sources": [],
            "answer": "",
            "error": None,
            "vectorstore": None
        }

        output = rag_pipeline.invoke(initial_state)

        print(f"❓ Question : {output['question']}")
        print(f"\n💬 Answer  :\n{output['answer']}")
        print(f"\n📎 Sources  : {list(set(output['sources']))}")
        print(f"\n📄 Chunks   : {len(output['chunks'])} retrieved")

        if output.get("error"):
            print(f"\n⚠️  Error   : {output['error']}")
        else:
            print("\n✅ Pipeline ran successfully with no errors")

    except Exception as e:
        print(f"❌ Pipeline execution failed: {e}")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except AttributeError as e:
    print(f"❌ Missing builtins object: {e}")

except KeyError as e:
    print(f"❌ Missing environment variable: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ LangGraph RAG pipeline compiled successfully
   Nodes : retrieve → (generate | error_handler) → END

🚀 Running LangGraph pipeline …

❓ Question : What is a large language model?

💬 Answer  :
A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. [Source: Large language model]

📎 Sources  : ['Large language model', 'Retrieval-augmented generation']

📄 Chunks   : 5 retrieved

✅ Pipeline ran successfully with no errors


In [8]:
# CELL 8 — RAGAS-style evaluation using Groq

try:
    import pandas as pd
    import time
    from groq import Groq
    import os

    groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

    # ── 20 test questions ─────────────────────────────────────────────────────────
    TEST_QUESTIONS = [
        "What is retrieval-augmented generation?",
        "What is a large language model?",
        "How do transformers work in deep learning?",
        "What is word embedding in NLP?",
        "Explain convolutional neural networks.",
        "What is a recurrent neural network?",
        "How does the attention mechanism work in machine learning?",
        "What is BERT and what is it used for?",
        "What are the key features of GPT-4?",
        "What is a knowledge graph?",
        "What is a vector database?",
        "What is reinforcement learning from human feedback?",
        "How does chunking help in RAG systems?",
        "What role do embeddings play in retrieval systems?",
        "What is the difference between semantic search and keyword search?",
        "How does RAG reduce hallucinations in AI systems?",
        "What are the main components of a transformer architecture?",
        "How is context precision measured in RAG evaluation?",
        "What is the purpose of a vector store in a RAG pipeline?",
        "How does self-attention differ from traditional attention?",
    ]

    # ── Helper: score one answer with Groq ───────────────────────────────────────
    def score_answer(question: str, answer: str, context: str) -> dict:
        """
        Ask Groq to score faithfulness and answer relevancy on a 0-1 scale.
        Returns dict with faithfulness, answer_relevancy, context_precision.
        """

        scoring_prompt = f"""You are an expert RAG evaluator. Score the following RAG output strictly.

QUESTION: {question}

RETRIEVED CONTEXT:
{context[:1500]}

GENERATED ANSWER:
{answer[:800]}

Score each metric from 0.0 to 1.0 (two decimal places):
1. faithfulness       — Is the answer fully supported by the context? (1.0 = fully grounded, 0.0 = hallucinated)
2. answer_relevancy   — Does the answer directly address the question? (1.0 = perfectly relevant, 0.0 = off-topic)
3. context_precision  — Does the retrieved context contain information needed to answer? (1.0 = highly relevant context, 0.0 = irrelevant)

Reply in this EXACT format and nothing else:
faithfulness: <score>
answer_relevancy: <score>
context_precision: <score>"""

        for attempt in range(3):
            try:
                response = groq_client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[{"role": "user", "content": scoring_prompt}],
                    temperature=0.0,
                    max_tokens=100
                )

                raw = response.choices[0].message.content.strip()

                # Parse the three scores
                scores = {}

                for line in raw.splitlines():
                    if "faithfulness:" in line:
                        scores["faithfulness"] = float(line.split(":")[1].strip())

                    elif "answer_relevancy:" in line:
                        scores["answer_relevancy"] = float(line.split(":")[1].strip())

                    elif "context_precision:" in line:
                        scores["context_precision"] = float(line.split(":")[1].strip())

                if len(scores) == 3:
                    return scores

            except Exception as e:
                if attempt < 2:
                    time.sleep(10)
                else:
                    print(f"❌ Scoring failed after 3 attempts: {e}")

        # Fallback if parsing fails
        return {
            "faithfulness": 0.0,
            "answer_relevancy": 0.0,
            "context_precision": 0.0
        }

    # ── Run pipeline + scoring for all 20 questions ───────────────────────────────
    print("🔄 Running RAG pipeline + scoring for 20 questions …\n")

    records = []

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(
            f"  [{i:02d}/20] {question[:65]}{'...' if len(question) > 65 else ''}",
            end=" "
        )

        try:
            # Step 1: RAG answer
            rag_result = ask_question(question, k=5)
            answer = rag_result["answer"]
            chunks = rag_result["chunks"]
            sources = rag_result["sources"]
            context = "\n\n".join(chunks)

            # Step 2: Score with Groq
            scores = score_answer(question, answer, context)

            records.append({
                "question": question,
                "answer": answer[:120] + "..." if len(answer) > 120 else answer,
                "sources": ", ".join(set(sources)),
                "faithfulness": scores["faithfulness"],
                "answer_relevancy": scores["answer_relevancy"],
                "context_precision": scores["context_precision"],
            })

            print(
                f"✅  F={scores['faithfulness']:.2f}  "
                f"AR={scores['answer_relevancy']:.2f}  "
                f"CP={scores['context_precision']:.2f}"
            )

        except Exception as e:
            print(f"❌ Error: {e}")

            records.append({
                "question": question,
                "answer": "Error",
                "sources": "",
                "faithfulness": 0.0,
                "answer_relevancy": 0.0,
                "context_precision": 0.0,
            })

        time.sleep(1)   # avoid Groq rate limit

    # ── Build results DataFrame ───────────────────────────────────────────────────
    try:
        df = pd.DataFrame(records)
        df.index = df.index + 1   # start index at 1

        print("\n" + "=" * 90)
        print("📊 RAGAS-STYLE EVALUATION RESULTS — 20 TEST QUESTIONS")
        print("=" * 90)

        print(
            df[
                [
                    "question",
                    "faithfulness",
                    "answer_relevancy",
                    "context_precision"
                ]
            ].to_string()
        )

        print("\n" + "=" * 90)
        print("📈 AGGREGATE SCORES")
        print("=" * 90)

        print(f"  Faithfulness        : {df['faithfulness'].mean():.3f}")
        print(f"  Answer Relevancy    : {df['answer_relevancy'].mean():.3f}")
        print(f"  Context Precision   : {df['context_precision'].mean():.3f}")

        print(
            f"  Overall Average     : "
            f"{df[['faithfulness','answer_relevancy','context_precision']].mean().mean():.3f}"
        )

    except Exception as e:
        print(f"❌ Error creating evaluation DataFrame: {e}")
        raise

    # Save for Cell 9
    try:
        import builtins

        builtins.eval_df = df
        print("\n✅ Results saved — proceed to Cell 9")

    except Exception as e:
        print(f"❌ Error saving results to builtins: {e}")

except ImportError as e:
    print(f"❌ Import Error: {e}")

except KeyError as e:
    print(f"❌ Missing Environment Variable: {e}")

except NameError as e:
    print(f"❌ Required object not found: {e}")

except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

🔄 Running RAG pipeline + scoring for 20 questions …

  [01/20] What is retrieval-augmented generation? ✅  F=1.00  AR=1.00  CP=1.00
  [02/20] What is a large language model? ✅  F=1.00  AR=1.00  CP=1.00
  [03/20] How do transformers work in deep learning? ✅  F=0.90  AR=0.95  CP=0.98
  [04/20] What is word embedding in NLP? ✅  F=1.00  AR=1.00  CP=1.00
  [05/20] Explain convolutional neural networks. ✅  F=1.00  AR=1.00  CP=1.00
  [06/20] What is a recurrent neural network? ✅  F=1.00  AR=1.00  CP=1.00
  [07/20] How does the attention mechanism work in machine learning? ✅  F=0.90  AR=0.95  CP=0.95
  [08/20] What is BERT and what is it used for? ✅  F=1.00  AR=1.00  CP=1.00
  [09/20] What are the key features of GPT-4? ✅  F=0.80  AR=1.00  CP=0.90
  [10/20] What is a knowledge graph? ✅  F=1.00  AR=1.00  CP=1.00
  [11/20] What is a vector database? ✅  F=1.00  AR=1.00  CP=1.00
  [12/20] What is reinforcement learning from human feedback? ✅  F=1.00  AR=1.00  CP=1.00
  [13/20] How does chunking hel

In [ ]:
# CELL 9 — Display final evaluation table

import builtins
import pandas as pd

df = builtins.eval_df

print("=" * 90)
print("📊 FULL EVALUATION TABLE")
print("=" * 90)
print(df.to_string())

print("\n" + "=" * 90)
print("📈 AGGREGATE SCORES")
print("=" * 90)
print(f"  Faithfulness        : {df['faithfulness'].mean():.3f}")
print(f"  Answer Relevancy    : {df['answer_relevancy'].mean():.3f}")
print(f"  Context Precision   : {df['context_precision'].mean():.3f}")
print(f"  Overall Average     : {df[['faithfulness','answer_relevancy','context_precision']].mean().mean():.3f}")

# Highlight best and worst
best  = df.loc[df['faithfulness'].idxmax(), 'question']
worst = df.loc[df['faithfulness'].idxmin(), 'question']
print(f"\n  🏆 Best  faithfulness : {best[:70]}")
print(f"  ⚠️  Worst faithfulness : {worst[:70]}")

📊 FULL EVALUATION TABLE
                                                              question                                                                                                                        answer                                                                                        sources  faithfulness  answer_relevancy  context_precision
1                              What is retrieval-augmented generation?   Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporat...                                                Vector database, Retrieval-augmented generation           1.0              1.00               1.00
2                                      What is a large language model?   A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks,...                                           Retrieval-augmented generation, Large language mode